# 03 -- Strategie 1 : Modification de la Fonction de Perte

| | |
|---|---|
| **Projet** | Gestion du Desequilibre de Classes pour l'Analyse de Sentiments en Darija Algerien |
| **Etudiant** | Abdelaziz Merzoug |
| **Module** | Machine Learning -- Master 1 DS & NLP -- USDB Blida 1 |
| **Encadrante** | Dr. Soraya Cheriguene |
| **Semaine** | 3 (22-29 Mars 2026) |
| **Plateforme** | Google Colab -- GPU T4 |

---

## Objectifs de ce notebook

On ne modifie **pas** les donnees : le train set reste desequilibre.
On change **uniquement** la fonction de perte pour corriger le biais.

**4 variantes testees** (chacune recharge DziriBERT from scratch) :

| Variante | Description |
|----------|-------------|
| A | Class Weighting -- poids inversement proportionnels a la frequence |
| B1 | Focal Loss gamma=1 (sans class weighting) |
| B2 | Focal Loss gamma=2 (sans class weighting) |
| C | Combinaison Class Weighting + Focal Loss gamma=2 |

**Memes hyperparametres que la Baseline** : epochs=5, lr=2e-5, batch_size=16, seed=42, AdamW.

**Metrique principale** : F1-macro (jamais Accuracy).

In [ ]:
# =============================================================================
# CELLULE 0 — Montage Google Drive + verification GPU
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mini_projet_darija'

import torch, os
assert torch.cuda.is_available(), 'GPU NON DISPONIBLE — changer le runtime !'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Verifier que les fichiers critiques de NB02 sont sur Drive
assert os.path.exists(f'{BASE}/data/split_indices.json'), \
    'split_indices.json MANQUANT sur Drive ! Relancer NB02 d abord.'
assert os.path.exists(f'{BASE}/results/baseline_metrics.json'), \
    'baseline_metrics.json MANQUANT sur Drive ! Relancer NB02 d abord.'

print(f'[OK] split_indices.json present sur Drive')
print(f'[OK] baseline_metrics.json present sur Drive')
print(f'[OK] Pret pour NB03')

In [ ]:
# =============================================================================
# CELLULE 1 — Installation + graines + imports + constantes
# =============================================================================
!pip install -q transformers datasets torch scikit-learn imbalanced-learn \
    pandas numpy matplotlib seaborn emoji accelerate

# Graines aleatoires -- fixees AVANT tout import aleatoire
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Reproductibilite CUDA
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Imports
import re
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji

from datasets import load_dataset
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    average_precision_score, accuracy_score
)
from sklearn.preprocessing import label_binarize
from imblearn.metrics import geometric_mean_score

import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)

# Constantes nommees -- identiques dans TOUS les notebooks
TEXT_COL   = 'Post'
LABEL_COL  = 'Polarity Class'
LANG_COL   = 'lang'
SEED       = 42

MODEL_NAME = 'alger-ia/dziribert'
EPOCHS     = 5
LR         = 2e-5
BATCH_SIZE = 16

LABEL_MAP   = {'Positive': 0, 'Negative': 1, 'Neutral': 2}
LABEL_NAMES = ['Positive', 'Negative', 'Neutral']

# Chemin Drive
BASE = '/content/drive/MyDrive/mini_projet_darija'

# Verification GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

if device.type == 'cuda':
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu_props.name}")
    print(f"VRAM   : {gpu_props.total_memory / 1e9:.1f} Go")

# Creation des repertoires locaux
for d in ['data', 'models', 'results', 'figures', 'logs']:
    os.makedirs(d, exist_ok=True)

print("\nEnvironnement configure avec succes.")

In [ ]:
# =============================================================================
# CELLULE 2 — Copier les fichiers critiques depuis Drive vers local
# =============================================================================
# split_indices.json : indices figes du NB02 -- JAMAIS re-split
shutil.copy(f'{BASE}/data/split_indices.json', 'data/split_indices.json')
print('split_indices.json copie depuis Drive.')

# baseline_metrics.json : score de reference pour comparaison
shutil.copy(f'{BASE}/results/baseline_metrics.json', 'results/baseline_metrics.json')
print('baseline_metrics.json copie depuis Drive.')

# Verification rapide
with open('data/split_indices.json') as f:
    si = json.load(f)
n_total = len(si['train_indices']) + len(si['val_indices']) + len(si['test_indices'])
print(f'  Train: {len(si["train_indices"])} | Val: {len(si["val_indices"])} | Test: {len(si["test_indices"])} | Total: {n_total}')

with open('results/baseline_metrics.json') as f:
    bm = json.load(f)
print(f'  Baseline F1-macro = {bm["f1_macro"]:.4f}')

---

## Chargement des donnees et reconstruction des splits

On recharge le corpus TWIFL, on applique le **meme preprocessing** que le Notebook 02,
puis on reconstruit les splits train/val/test a partir de `split_indices.json`.

**Regle absolue** : on ne re-split JAMAIS les donnees. Les indices sont figes.

In [ ]:
# =============================================================================
# CELLULE 3 — Chargement du corpus TWIFL depuis HuggingFace
# =============================================================================
dataset = load_dataset('arbml/Twifil')
df = dataset['train'].to_pandas()

print(f"Shape initiale : {df.shape}")
print(f"Distribution initiale de '{LABEL_COL}' :")
print(df[LABEL_COL].value_counts())

In [ ]:
# =============================================================================
# CELLULE 4 — Preprocessing IDENTIQUE au Notebook 02
# =============================================================================

def remove_urls(text):
    """Supprime les URLs (http, https, www)."""
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_mentions(text):
    """Supprime les mentions Twitter (@username)."""
    return re.sub(r'@\w+', '', text)

def convert_emojis(text):
    """Convertit les emojis en texte descriptif anglais."""
    return emoji.demojize(text, delimiters=(' ', ' '))

def remove_hashtag_symbol(text):
    """Supprime le # mais garde le texte du hashtag."""
    return text.replace('#', '')

def normalize_whitespace(text):
    """Normalise les espaces multiples."""
    return re.sub(r'\s+', ' ', text).strip()

def normalize_repeated_chars(text):
    """Reduit les repetitions excessives (max 2 consecutifs)."""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def normalize_arabic_digits(text):
    """Convertit les chiffres arabes-indiens en chiffres occidentaux."""
    arabic_to_western = str.maketrans(
        '\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669',
        '0123456789'
    )
    return text.translate(arabic_to_western)

# Application sequentielle
df_clean = df[[TEXT_COL, LABEL_COL, LANG_COL]].copy()
df_clean[TEXT_COL] = df_clean[TEXT_COL].astype(str)

df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(remove_urls)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(remove_mentions)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(convert_emojis)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(remove_hashtag_symbol)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(normalize_whitespace)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(normalize_repeated_chars)
df_clean[TEXT_COL] = df_clean[TEXT_COL].apply(normalize_arabic_digits)

# Suppression des doublons
df_clean = df_clean.drop_duplicates(subset=TEXT_COL, keep='first').reset_index(drop=True)

# Suppression des tweets vides/quasi-vides (< 2 caracteres)
empty_mask = df_clean[TEXT_COL].str.strip().str.len() < 2
df_clean = df_clean[~empty_mask].reset_index(drop=True)

print(f"Shape apres preprocessing : {df_clean.shape}")
print(f"\nDistribution des classes :")
for cls in LABEL_NAMES:
    n = (df_clean[LABEL_COL] == cls).sum()
    print(f"  {cls:10s} : {n:4d} ({n/len(df_clean)*100:.1f}%)")

In [ ]:
# =============================================================================
# CELLULE 5 — Chargement des indices de split (JAMAIS re-split)
# =============================================================================
with open('data/split_indices.json', 'r') as f:
    split_indices = json.load(f)

train_indices = split_indices['train_indices']
val_indices   = split_indices['val_indices']
test_indices  = split_indices['test_indices']

# Reconstruction des splits a partir du DataFrame preprocesse
X_train = df_clean.loc[df_clean.index.isin(train_indices), TEXT_COL]
y_train = df_clean.loc[df_clean.index.isin(train_indices), LABEL_COL]
X_val   = df_clean.loc[df_clean.index.isin(val_indices), TEXT_COL]
y_val   = df_clean.loc[df_clean.index.isin(val_indices), LABEL_COL]
X_test  = df_clean.loc[df_clean.index.isin(test_indices), TEXT_COL]
y_test  = df_clean.loc[df_clean.index.isin(test_indices), LABEL_COL]

print(f"=== Verification des splits (charges depuis split_indices.json) ===")
print(f"Train : {len(X_train):5d} ({len(X_train)/len(df_clean)*100:.1f}%)")
print(f"Val   : {len(X_val):5d} ({len(X_val)/len(df_clean)*100:.1f}%)")
print(f"Test  : {len(X_test):5d} ({len(X_test)/len(df_clean)*100:.1f}%)")
print(f"Total : {len(X_train) + len(X_val) + len(X_test):5d}")

print(f"\n=== Distribution du train set ===")
train_dist = y_train.value_counts()
for cls in LABEL_NAMES:
    print(f"  {cls:10s} : {train_dist[cls]:4d} ({train_dist[cls]/len(y_train)*100:.1f}%)")

---

## Preparation des donnees et fonctions d'evaluation

In [ ]:
# =============================================================================
# CELLULE 6 — Tokenisation et Dataset PyTorch
# =============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TweetDataset(torch.utils.data.Dataset):
    """Dataset PyTorch pour les tweets tokenises."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=False,
            max_length=max_length, return_tensors=None
        )
        self.labels = [LABEL_MAP[l] for l in labels]

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenisation en cours...")
train_dataset = TweetDataset(X_train, y_train, tokenizer)
val_dataset   = TweetDataset(X_val, y_val, tokenizer)
test_dataset  = TweetDataset(X_test, y_test, tokenizer)

print(f"Train dataset : {len(train_dataset)} exemples")
print(f"Val dataset   : {len(val_dataset)} exemples")
print(f"Test dataset  : {len(test_dataset)} exemples")

In [ ]:
# =============================================================================
# CELLULE 7 — evaluate_model() copiee EXACTEMENT du NB02
# =============================================================================

def evaluate_model(y_true, y_pred, y_proba, class_names=None):
    """Calcule TOUTES les metriques imposees par l'Enonce."""
    if class_names is None:
        class_names = LABEL_NAMES
    results = {}
    # 1. F1-macro (METRIQUE PRINCIPALE)
    results['f1_macro'] = float(f1_score(y_true, y_pred, average='macro'))
    # 2. F1 par classe
    f1s = f1_score(y_true, y_pred, average=None)
    for i, name in enumerate(class_names):
        results[f'f1_{name.lower()}'] = float(f1s[i])
    # 3. Precision et Rappel par classe
    prec, rec, _, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    for i, name in enumerate(class_names):
        results[f'precision_{name.lower()}'] = float(prec[i])
        results[f'recall_{name.lower()}'] = float(rec[i])
    # 4. AUC-PR macro
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    auc_pr_per_class = []
    for i in range(3):
        ap = average_precision_score(y_true_bin[:, i], y_proba[:, i])
        auc_pr_per_class.append(ap)
        results[f'auc_pr_{class_names[i].lower()}'] = float(ap)
    results['auc_pr_macro'] = float(np.mean(auc_pr_per_class))
    # 5. G-mean
    results['g_mean'] = float(geometric_mean_score(y_true, y_pred, average='macro'))
    # 6. Accuracy (UNIQUEMENT pour illustrer le probleme)
    results['accuracy'] = float(accuracy_score(y_true, y_pred))
    # 7. Matrice de confusion
    results['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    # 8. Rapport complet
    results['classification_report'] = classification_report(
        y_true, y_pred, target_names=class_names, digits=4
    )
    return results


def print_metrics(results, config_name=""):
    """Affiche les metriques de facon lisible."""
    print(f"\n{'='*60}")
    print(f"  RESULTATS : {config_name}")
    print(f"{'='*60}")
    print(f"  F1-macro (PRINCIPAL) : {results['f1_macro']:.4f}")
    print(f"  Accuracy (ILLUSTR.)  : {results['accuracy']:.4f}")
    print(f"  AUC-PR macro         : {results['auc_pr_macro']:.4f}")
    print(f"  G-mean               : {results['g_mean']:.4f}")
    print(f"\n  F1 par classe :")
    for name in LABEL_NAMES:
        f1 = results[f'f1_{name.lower()}']
        p  = results[f'precision_{name.lower()}']
        r  = results[f'recall_{name.lower()}']
        print(f"    {name:10s} : F1={f1:.4f}  Prec={p:.4f}  Rec={r:.4f}")
    print(f"\n{results['classification_report']}")


def plot_confusion_matrix(results, config_name, save_path):
    """Affiche et sauvegarde la matrice de confusion."""
    cm = np.array(results['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel('Predit', fontsize=12)
    ax.set_ylabel('Vrai', fontsize=12)
    ax.set_title(f'Matrice de Confusion -- {config_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Matrice de confusion sauvegardee : {save_path}")


def compute_metrics(eval_pred):
    """Callback pour le Trainer -- F1-macro pendant l'entrainement."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds),
    }


def save_to_drive():
    """Copie tous les results/ et figures/ vers Google Drive."""
    for f in os.listdir('results'):
        shutil.copy(f'results/{f}', f'{BASE}/results/{f}')
    for f in os.listdir('figures'):
        if f.endswith('.png'):
            shutil.copy(f'figures/{f}', f'{BASE}/figures/{f}')
    print('Sauvegarde intermediaire sur Drive effectuee.')


print("Fonctions definies : evaluate_model, print_metrics, plot_confusion_matrix,")
print("                     compute_metrics, save_to_drive.")

In [ ]:
# =============================================================================
# CELLULE 8 — train_and_evaluate() avec sauvegarde Drive automatique
# =============================================================================

def train_and_evaluate(trainer_cls, trainer_kwargs, config_name, output_dir,
                       metrics_path, cm_path):
    """
    Lance un entrainement complet, evalue, sauvegarde, puis copie sur Drive.
    Le modele DziriBERT est recharge FROM SCRATCH a chaque appel.
    """
    # Reinitialiser les graines
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # Recharger DziriBERT FROM SCRATCH (obligatoire)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3
    )
    print(f"\nModele {MODEL_NAME} recharge from scratch pour : {config_name}")

    # Hyperparametres VERROUILLES
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        optim='adamw_torch',
        seed=SEED,
        data_seed=SEED,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        logging_dir=f'./logs/{output_dir.split("/")[-1]}',
        report_to='none',
        fp16=torch.cuda.is_available(),
        warmup_ratio=0.1,
        weight_decay=0.01,
        save_total_limit=2,
    )

    trainer = trainer_cls(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        **trainer_kwargs
    )

    print(f"Lancement : {config_name}")
    print(f"  Epochs={EPOCHS} | LR={LR} | Batch={BATCH_SIZE} | Seed={SEED} | Optim=AdamW")
    train_result = trainer.train()
    print(f"Entrainement termine en {train_result.metrics.get('train_runtime', 0):.0f}s")

    # Evaluation sur le test set FIGE
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    y_proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    y_pred = np.argmax(logits, axis=-1)
    y_true = np.array([LABEL_MAP[l] for l in y_test])

    results = evaluate_model(y_true, y_pred, y_proba)
    print_metrics(results, config_name)

    # Sauvegarde locale
    results_to_save = {k: v for k, v in results.items() if k != 'classification_report'}
    with open(metrics_path, 'w') as f:
        json.dump(results_to_save, f, indent=2)
    print(f"Metriques sauvegardees : {metrics_path}")

    plot_confusion_matrix(results, config_name, cm_path)

    # Sauvegarde IMMEDIATE sur Drive (protection deconnexion Colab)
    save_to_drive()

    # Liberer la memoire GPU
    del model, trainer
    torch.cuda.empty_cache()

    return results

print("Fonction train_and_evaluate() definie (avec sauvegarde Drive automatique).")

---

## Variante A -- Class Weighting

Formule imposee par l'Enonce (par. 4.1) :

$$\text{poids}_c = \frac{N_{total}}{K \times N_c}$$

Les erreurs sur Neutral (minoritaire) comptent davantage dans la perte totale.

In [ ]:
# =============================================================================
# CELLULE 9 — Calcul des poids de classe (formule IMPOSEE)
# =============================================================================
total_train = len(y_train)
nb_classes = 3
class_counts = y_train.value_counts()

weights = {}
for cls in LABEL_NAMES:
    weights[cls] = total_train / (nb_classes * class_counts[cls])
    print(f'Poids {cls:10s} : {weights[cls]:.4f}  (nb exemples = {class_counts[cls]})')

# Tenseur dans le MEME ORDRE que LABEL_MAP : Positive=0, Negative=1, Neutral=2
class_weights_tensor = torch.tensor(
    [weights['Positive'], weights['Negative'], weights['Neutral']],
    dtype=torch.float
).to(device)

print(f"\nTenseur des poids : {class_weights_tensor}")
print(f"Device du tenseur : {class_weights_tensor.device}")

assert weights['Neutral'] > weights['Negative'] > weights['Positive'], \
    "Les poids doivent refleter l'inverse de la frequence"
print("Verification OK : Neutral > Negative > Positive (poids corrects).")

In [ ]:
# =============================================================================
# CELLULE 10 — WeightedLossTrainer
# =============================================================================

class WeightedLossTrainer(Trainer):
    """Trainer avec CrossEntropyLoss ponderee par les poids de classe."""
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("WeightedLossTrainer defini.")

In [ ]:
# =============================================================================
# CELLULE 11 — VARIANTE A : Entrainement Class Weighting (~45 min)
# =============================================================================

cw_results = train_and_evaluate(
    trainer_cls=WeightedLossTrainer,
    trainer_kwargs={'class_weights': class_weights_tensor},
    config_name='Class Weighting',
    output_dir='./models/cw',
    metrics_path='results/cw_metrics.json',
    cm_path='figures/cw_confusion_matrix.png'
)

### Analyse immediate : degradation du Class Weighting

Le Class Weighting a **degrade** les performances par rapport a la Baseline :

| Métrique        | Baseline | Class Weighting | Delta        |
|-----------------|----------|-----------------|--------------|
| F1-macro        | 0.6805   | 0.6386          | **-0.0420**  |
| F1 Positive     | 0.7806   | 0.7141          | **-0.0665**  |
| F1 Neutral      | 0.5596   | 0.5134          | **-0.0462**  |
| Recall Positive | 0.8031   | 0.6457          | **-0.1575**  |
| Recall Neutral  | 0.5567   | 0.6907          | +0.1340      |

**Explication** : le preprocessing a aggrave le desequilibre de 2.10:1 (brut) a 3.93:1
(Positive=1778 vs Neutral=452 dans le train set). Avec cette distribution, la formule
impose des poids tres asymetriques (Positive=0.64, Neutral=2.54). Le modele sur-predit
Neutral au detriment de Positive : le recall Positive chute de 0.80 a 0.65, ce qui
signifie que 135 tweets Positifs sont desormais mal classes (visible dans la matrice de
confusion : 74 predits Negative + 61 predits Neutral au lieu de Positive).

Le gain en recall Neutral (+0.13) ne compense pas la perte massive en Positive.
Le F1-macro, qui moyenne les F1 sans ponderation, sanctionne cette sur-correction.

**Conclusion partielle** : sur un corpus ou le desequilibre post-preprocessing est
de 3.93:1, le Class Weighting statique est trop agressif. Les strategies suivantes
(Focal Loss) offrent une correction plus nuancee.

---

## Variante B -- Focal Loss

$$\text{FL}(p_t) = -(1 - p_t)^\gamma \cdot \log(p_t)$$

gamma=1 et gamma=2 sont des experiences **completement separees**.

Reference : [4] T. Y. Lin et al., "Focal Loss for Dense Object Detection", IEEE ICCV, 2017.

In [ ]:
# =============================================================================
# CELLULE 12 — Implementation FocalLoss + FocalLossTrainer
# =============================================================================

class FocalLoss(nn.Module):
    """Focal Loss pour classification multi-classes.
    gamma controle la reduction des exemples faciles.
    alpha (optionnel) : poids de classe pour la combinaison CW+FL."""

    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.alpha, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class FocalLossTrainer(Trainer):
    """Trainer avec Focal Loss."""
    def __init__(self, focal_loss_fn, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.focal_loss_fn.alpha is not None:
            self.focal_loss_fn.alpha = self.focal_loss_fn.alpha.to(logits.device)
        loss = self.focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("FocalLoss et FocalLossTrainer definis.")

### Variante B1 -- Focal Loss gamma=1 (sans class weighting)

In [ ]:
# =============================================================================
# CELLULE 13 — VARIANTE B1 : Focal Loss gamma=1 (~45 min)
# =============================================================================

fl_g1_loss = FocalLoss(gamma=1.0, alpha=None)
print(f"Focal Loss : gamma={fl_g1_loss.gamma}, alpha={fl_g1_loss.alpha}")

fl_g1_results = train_and_evaluate(
    trainer_cls=FocalLossTrainer,
    trainer_kwargs={'focal_loss_fn': fl_g1_loss},
    config_name='Focal Loss (gamma=1)',
    output_dir='./models/fl_g1',
    metrics_path='results/fl_g1_metrics.json',
    cm_path='figures/fl_g1_confusion_matrix.png'
)

### Variante B2 -- Focal Loss gamma=2 (experience SEPAREE de B1)

In [ ]:
# =============================================================================
# CELLULE 14 — VARIANTE B2 : Focal Loss gamma=2 (~45 min)
# Experience COMPLETEMENT SEPAREE de B1.
# =============================================================================

fl_g2_loss = FocalLoss(gamma=2.0, alpha=None)
print(f"Focal Loss : gamma={fl_g2_loss.gamma}, alpha={fl_g2_loss.alpha}")

fl_g2_results = train_and_evaluate(
    trainer_cls=FocalLossTrainer,
    trainer_kwargs={'focal_loss_fn': fl_g2_loss},
    config_name='Focal Loss (gamma=2)',
    output_dir='./models/fl_g2',
    metrics_path='results/fl_g2_metrics.json',
    cm_path='figures/fl_g2_confusion_matrix.png'
)

---

## Variante C -- Combinaison Class Weighting + Focal Loss (gamma=2)

On passe le tenseur `alpha` (poids de classe) dans la Focal Loss.
Reference Enonce par. 4.3.

In [ ]:
# =============================================================================
# CELLULE 15 — VARIANTE C : CW + Focal Loss gamma=2 (~45 min)
# =============================================================================

fl_cw_loss = FocalLoss(gamma=2.0, alpha=class_weights_tensor.clone())
print(f"Focal Loss : gamma={fl_cw_loss.gamma}")
print(f"  alpha (class weights) : {fl_cw_loss.alpha}")

cw_fl_results = train_and_evaluate(
    trainer_cls=FocalLossTrainer,
    trainer_kwargs={'focal_loss_fn': fl_cw_loss},
    config_name='CW + Focal Loss (gamma=2)',
    output_dir='./models/cw_fl',
    metrics_path='results/cw_fl_metrics.json',
    cm_path='figures/cw_fl_confusion_matrix.png'
)

---

## Tableau comparatif : Baseline vs 4 variantes

In [ ]:
# =============================================================================
# CELLULE 16 — Chargement baseline + construction du tableau comparatif
# =============================================================================
with open('results/baseline_metrics.json', 'r') as f:
    baseline_results = json.load(f)

all_results = {
    'Baseline': baseline_results,
    'Class Weighting': cw_results,
    'Focal Loss (gamma=1)': fl_g1_results,
    'Focal Loss (gamma=2)': fl_g2_results,
    'CW + Focal Loss': cw_fl_results,
}

table_data = []
for config_name, res in all_results.items():
    table_data.append({
        'Configuration': config_name,
        'F1-macro': f"{res['f1_macro']:.4f}",
        'F1 Positive': f"{res['f1_positive']:.4f}",
        'F1 Negative': f"{res['f1_negative']:.4f}",
        'F1 Neutral': f"{res['f1_neutral']:.4f}",
        'AUC-PR': f"{res['auc_pr_macro']:.4f}",
        'G-mean': f"{res['g_mean']:.4f}",
        'Accuracy': f"{res['accuracy']:.4f}",
    })

df_comparison = pd.DataFrame(table_data)
print("\n" + "="*100)
print("  TABLEAU COMPARATIF -- STRATEGIE 1")
print("="*100)
print(df_comparison.to_string(index=False))

df_comparison.to_csv('results/strategie1_comparatif.csv', index=False)
print("\nTableau sauvegarde : results/strategie1_comparatif.csv")

best_config = max(all_results.items(), key=lambda x: x[1]['f1_macro'])
print(f"\nMeilleure configuration : {best_config[0]} "
      f"(F1-macro = {best_config[1]['f1_macro']:.4f})")

In [ ]:
# =============================================================================
# CELLULE 17 — Bar chart F1-macro pour les 5 configurations
# =============================================================================
configs = list(all_results.keys())
f1_macros = [all_results[c]['f1_macro'] for c in configs]
colors = ['#888888', '#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(configs, f1_macros, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, f1_macros):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.axhline(y=baseline_results['f1_macro'], color='red', linestyle='--',
           linewidth=1.5, label=f"Baseline ({baseline_results['f1_macro']:.4f})")
ax.set_ylabel('F1-macro', fontsize=13)
ax.set_title('Strategie 1 -- F1-macro par configuration', fontsize=14)
ax.set_ylim(0, max(f1_macros) * 1.15)
ax.legend(fontsize=11)
plt.xticks(rotation=15, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('figures/strategie1_f1_macro_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : figures/strategie1_f1_macro_comparison.png")

In [ ]:
# =============================================================================
# CELLULE 18 — F1 par classe pour chaque configuration
# =============================================================================
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(configs))
width = 0.25
f1_pos = [all_results[c]['f1_positive'] for c in configs]
f1_neg = [all_results[c]['f1_negative'] for c in configs]
f1_neu = [all_results[c]['f1_neutral'] for c in configs]
ax.bar(x - width, f1_pos, width, label='Positive', color='#4C72B0')
ax.bar(x, f1_neg, width, label='Negative', color='#C44E52')
ax.bar(x + width, f1_neu, width, label='Neutral', color='#55A868')
ax.set_ylabel('F1-score', fontsize=13)
ax.set_title('Strategie 1 -- F1 par classe et par configuration', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=15, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('figures/strategie1_f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : figures/strategie1_f1_per_class.png")

In [ ]:
# =============================================================================
# CELLULE 19 — Matrices de confusion cote a cote : Baseline vs meilleure
# =============================================================================
best_name = best_config[0]
best_res  = best_config[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm_baseline = np.array(baseline_results['confusion_matrix'])
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_xlabel('Predit', fontsize=12)
axes[0].set_ylabel('Vrai', fontsize=12)
axes[0].set_title(f'Baseline (F1-macro={baseline_results["f1_macro"]:.4f})', fontsize=13)

cm_best = np.array(best_res['confusion_matrix'])
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
axes[1].set_xlabel('Predit', fontsize=12)
axes[1].set_ylabel('Vrai', fontsize=12)
axes[1].set_title(f'{best_name} (F1-macro={best_res["f1_macro"]:.4f})', fontsize=13)

plt.suptitle('Comparaison : Baseline vs Meilleure Variante (Strategie 1)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/strategie1_cm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : figures/strategie1_cm_comparison.png")

In [ ]:
# =============================================================================
# AUC-PR par classe pour chaque configuration
# L'AUC-PR (Average Precision) est plus informative que l'AUC-ROC sur
# des donnees desequilibrees car elle est sensible au ratio de classes.
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(configs))
width = 0.25

auc_pos = [all_results[c]['auc_pr_positive'] for c in configs]
auc_neg = [all_results[c]['auc_pr_negative'] for c in configs]
auc_neu = [all_results[c]['auc_pr_neutral'] for c in configs]

ax.bar(x - width, auc_pos, width, label='AUC-PR Positive', color='#4C72B0')
ax.bar(x, auc_neg, width, label='AUC-PR Negative', color='#C44E52')
ax.bar(x + width, auc_neu, width, label='AUC-PR Neutral', color='#55A868')

ax.set_ylabel('AUC-PR', fontsize=13)
ax.set_title('Strategie 1 -- AUC-PR par classe et par configuration', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=15, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('figures/strategie1_auc_pr_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : figures/strategie1_auc_pr_per_class.png")
plt.close('all')

In [ ]:
# =============================================================================
# CELLULE 20 — Pourquoi l'Accuracy est trompeuse
# =============================================================================
print("=" * 70)
print("  POURQUOI L'ACCURACY EST TROMPEUSE SUR TWIFL")
print("=" * 70)
for config_name, res in all_results.items():
    acc = res['accuracy']
    f1  = res['f1_macro']
    delta = acc - f1
    print(f"  {config_name:25s} : Accuracy={acc:.4f}  F1-macro={f1:.4f}  "
          f"(ecart={delta:+.4f})")
print()
print("Observation : l'Accuracy surestime les performances en masquant")
print("les erreurs sur les classes minoritaires (Negative, Neutral).")
print("C'est pourquoi le F1-macro est notre METRIQUE PRINCIPALE.")

---

## Analyse obligatoire -- 4 questions (Enonce par. 4.4)

In [ ]:
# =============================================================================
# CELLULE 21 — Calcul des deltas pour l'analyse
# =============================================================================
baseline_f1_macro = baseline_results['f1_macro']
baseline_f1_neu   = baseline_results['f1_neutral']
baseline_f1_pos   = baseline_results['f1_positive']

print("=" * 70)
print("  DONNEES POUR L'ANALYSE")
print("=" * 70)
for config_name, res in all_results.items():
    if config_name == 'Baseline':
        continue
    delta_macro = res['f1_macro'] - baseline_f1_macro
    delta_neu   = res['f1_neutral'] - baseline_f1_neu
    delta_pos   = res['f1_positive'] - baseline_f1_pos
    print(f"\n{config_name} :")
    print(f"  F1-macro           : {res['f1_macro']:.4f} (delta={delta_macro:+.4f})")
    print(f"  F1 Neutral (minor) : {res['f1_neutral']:.4f} (delta={delta_neu:+.4f})")
    print(f"  F1 Positive (major): {res['f1_positive']:.4f} (delta={delta_pos:+.4f})")

### Question 1 : Le F1 de la classe minoritaire a-t-il augmente par rapport a la Baseline ? De combien ?

**Reponse** :

Non. Aucune variante n'ameliore significativement le F1-Neutral. Le Baseline obtient F1-Neutral = 0.5596. Le Class Weighting obtient 0.5134 (delta = -0.0462), la Focal Loss gamma=1 obtient 0.5525 (delta = -0.0071), la Focal Loss gamma=2 obtient 0.5543 (delta = -0.0053), et la combinaison CW+FL obtient 0.5446 (delta = -0.0150). Le Class Weighting degrade le plus fortement le F1-Neutral car les poids agressifs (ratio 3.93:1) sur-corrigent et destabilisent l'apprentissage de la classe minoritaire.

---

### Question 2 : Y a-t-il un compromis ? Le F1 de la classe majoritaire a-t-il baisse quand celui de la minoritaire a augmente ?

**Reponse** :

Oui, un compromis est observe avec le Class Weighting : le F1-Positive chute de 0.7806 (Baseline) a 0.7141 (delta = -0.0665) tandis que le recall Neutral augmente de 0.5567 a 0.6907. Cependant, ce gain de recall ne compense pas la perte de precision Neutral (de 0.5625 a 0.4085), d'ou la chute du F1-Neutral. Les variantes Focal Loss maintiennent le F1-Positive proche du Baseline (0.7614 pour gamma=1, 0.7605 pour gamma=2), montrant un compromis beaucoup plus doux.

---

### Question 3 : Quel gamma donne les meilleurs resultats sur TWIFL ? Pouvez-vous expliquer pourquoi ?

**Reponse** :

Gamma=2 obtient F1-macro = 0.6794, marginalement superieur a gamma=1 (F1-macro = 0.6784, delta = +0.0010). La difference est statistiquement insignifiante. Sur TWIFL, avec un desequilibre de 3.93:1, les deux valeurs de gamma offrent des corrections comparables. Gamma=2 montre un leger avantage sur le recall Negative (0.7692 vs 0.7615) et le F1-Neutral (0.5543 vs 0.5525), suggerant qu'une reduction plus agressive des exemples faciles beneficie legerement aux classes minoritaires.

---

### Question 4 : Class Weighting ou Focal Loss -- quelle variante est la plus efficace sur ce corpus ?

**Reponse** :

La Focal Loss (gamma=2) est la variante la plus efficace avec F1-macro = 0.6794, quasiment identique au Baseline (0.6805, delta = -0.0011). Le Class Weighting est la moins efficace (F1-macro = 0.6386, delta = -0.0419). La combinaison CW+FL (0.6683) est inferieure a FL seule (0.6794), confirmant une interference entre les deux mecanismes de correction. Conclusion : sur un desequilibre modere (3.93:1), la Focal Loss seule est preferable au Class Weighting car elle ajuste dynamiquement les poids par exemple plutot qu'une correction statique par classe.

---

## Synthese -- Points cles pour le rapport

In [ ]:
# =============================================================================
# CELLULE 22 — Points cles pour le rapport
# =============================================================================
print("=" * 70)
print("  POINTS CLES -- STRATEGIE 1 : MODIFICATION DE LA FONCTION DE PERTE")
print("=" * 70)

print(f"\n1. BASELINE DE REFERENCE")
print(f"   F1-macro : {baseline_results['f1_macro']:.4f}")
print(f"   F1 Neutral (classe la plus faible) : {baseline_results['f1_neutral']:.4f}")

print(f"\n2. RESULTATS PAR VARIANTE")
for config_name, res in all_results.items():
    if config_name == 'Baseline':
        continue
    delta = res['f1_macro'] - baseline_f1_macro
    sign = '+' if delta >= 0 else ''
    print(f"   {config_name:25s} : F1-macro={res['f1_macro']:.4f} "
          f"({sign}{delta:.4f} vs baseline)")

best_name = best_config[0]
best_f1 = best_config[1]['f1_macro']
print(f"\n3. CONCLUSION : Aucune variante de Strategie 1 ne depasse le Baseline.")
print(f"   Plus proche du Baseline : Focal Loss gamma=2 (F1-macro=0.6794, delta=-0.0011).")

print(f"\n4. CONCLUSION")
print(f"   La modification de la fonction de perte corrige partiellement")
print(f"   le biais vers la classe majoritaire SANS modifier les donnees.")

---

## Synthese -- Points cles pour le rapport
**1. Baseline de reference** : F1-macro = 0.6805, F1 Neutral = 0.5596. Le modele est biaise vers Positive (F1 = 0.7806) au detriment de Neutral.

**2. Class Weighting** : F1-macro = 0.6386 (**-0.0419 vs Baseline**). Degradation causee par des poids trop agressifs (Neutral x2.54) sur un desequilibre aggrave a 3.93:1 apres preprocessing. Le recall Neutral augmente (+0.134) mais la precision chute (-0.154), resultant en un F1 Neutral pire (0.5134).

**3. Focal Loss** : gamma=1 (F1-macro=0.6784) et gamma=2 (F1-macro=0.6794) produisent des resultats quasi identiques et proches de la Baseline. La correction dynamique est plus stable que le Class Weighting mais insuffisante pour ameliorer la classe Neutral.

**4. Combinaison CW + Focal Loss** : F1-macro = 0.6683 (**-0.0122 vs Baseline**). La double correction amplifie les effets negatifs du Class Weighting sans les compenser.

**5. Conclusion Strategie 1** : La modification de la fonction de perte ne suffit pas sur TWIFL. Le plafond du F1 Neutral a ~0.55 est un probleme de representation (manque de donnees, ambiguite semantique) et non de ponderation. Les Strategies 2 et 3 devront agir sur les donnees elles-memes.

**6. Note sur l'AUC-PR** : FL gamma=1 obtient l'AUC-PR la plus elevee (0.7687 vs
0.7647 pour gamma=2), mais un F1-macro inferieur (0.6784 vs 0.6794). Cette inversion
illustre la difference fondamentale entre une metrique a seuil fixe (F1) et une
metrique integree sur tous les seuils (AUC-PR). Le protocole impose le F1-macro comme
metrique principale : FL gamma=2 est donc la meilleure variante.

In [ ]:
# =============================================================================
# CELLULE 23 — Sauvegarde FINALE sur Google Drive
# =============================================================================
# Copier TOUS les resultats
for f in os.listdir('results'):
    shutil.copy(f'results/{f}', f'{BASE}/results/{f}')
    print(f'  [OK] results/{f}')

# Copier TOUTES les figures
for f in os.listdir('figures'):
    if f.endswith('.png'):
        shutil.copy(f'figures/{f}', f'{BASE}/figures/{f}')
        print(f'  [OK] figures/{f}')

print('\n=== SAUVEGARDE NB03 TERMINEE ===')

In [ ]:
# =============================================================================
# CELLULE 24 — Verification finale sur Drive
# =============================================================================
metriques_attendues = [
    'results/cw_metrics.json',
    'results/fl_g1_metrics.json',
    'results/fl_g2_metrics.json',
    'results/cw_fl_metrics.json',
    'results/strategie1_comparatif.csv',
]

figures_attendues = [
    'figures/cw_confusion_matrix.png',
    'figures/fl_g1_confusion_matrix.png',
    'figures/fl_g2_confusion_matrix.png',
    'figures/cw_fl_confusion_matrix.png',
    'figures/strategie1_f1_macro_comparison.png',
    'figures/strategie1_f1_per_class.png',
    'figures/strategie1_cm_comparison.png',
]

print('=== VERIFICATION NB03 ===')
all_ok = True
for f in metriques_attendues:
    p = f'{BASE}/{f}'
    if os.path.exists(p):
        if f.endswith('.json'):
            with open(p) as fp:
                m = json.load(fp)
            print(f'  [OK] {f} -- F1-macro = {m.get("f1_macro", "N/A")}')
        else:
            print(f'  [OK] {f}')
    else:
        print(f'  [MANQUANT] {f}')
        all_ok = False

for f in figures_attendues:
    p = f'{BASE}/{f}'
    status = 'OK' if os.path.exists(p) else 'MANQUANT'
    if status == 'MANQUANT':
        all_ok = False
    print(f'  [{status}] {f}')

if all_ok:
    print('\nTous les fichiers NB03 sont presents sur Drive.')
else:
    print('\nATTENTION : certains fichiers sont manquants !')